# Stage 5: BioMedCLIP zero-shot on NIH ChestX-ray14

Retrains the winning arm (lr 1e-5) with the epoch budget cut from 15 to 5, then runs both
evaluations in one session.

The open question: our best NIH Macro AUC is **0.589** with the `patient` prompt, on the
CLIP+ClinicalBERT checkpoint. Does the medical-pretrained backbone move it, and does the prompt
ablation pattern (`patient` best, `ensemble` and `pos_neg` worse) survive a backbone change? If it
does, that pattern stops being a quirk of one model.

**Requires:** GPU, Internet, and the `nih-chest-xrays/data` input attached (45GB -- too big to pull
with kagglehub). OpenI is optional; it falls back to kagglehub.

**Save Version before closing this tab** -- an idle session is reclaimed and `/kaggle/working` goes
with it.

In [ ]:
!pip install -q 'open_clip_torch>=3.0' transformers
!git clone -q https://github.com/maxzhang646/medical-clip.git
import sys
sys.path.insert(0, 'medical-clip/src')

import torch
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
from pathlib import Path

def find_dir(root, marker):
    root = Path(root)
    if not root.exists():
        return None
    if (root / marker).exists():
        return root
    for hit in root.rglob(marker):
        return hit.parent
    return None

INDIANA_DIR = find_dir('/kaggle/input', 'indiana_reports.csv')
if INDIANA_DIR is None:
    print('No attached OpenI input -- downloading with kagglehub ...', flush=True)
    import kagglehub
    INDIANA_DIR = find_dir(kagglehub.dataset_download('raddar/chest-xrays-indiana-university'),
                           'indiana_reports.csv')
NIH_DIR = find_dir('/kaggle/input', 'Data_Entry_2017.csv')

assert INDIANA_DIR, 'OpenI not found'
assert NIH_DIR, ('NIH not found. Add Input -> nih-chest-xrays/data. It is 45GB, so unlike OpenI '
                 'it cannot be pulled with kagglehub inside the session.')
print('INDIANA_DIR =', INDIANA_DIR)
print('NIH_DIR     =', NIH_DIR)

SPLIT_DIR = Path('medical-clip/splits')

## 1. Retrain, 5 epochs (~27 min)

The 15-epoch run peaked at epoch 3 and finished above the ln(64) = 4.159 random baseline. Cutting to
5 also changes the cosine schedule, so this is a new configuration rather than a reproduction —
it gets its own row in the results table.

In [ ]:
import subprocess, time

t0 = time.time()
proc = subprocess.Popen(
    ['python3', 'src/train.py', '--config', 'configs/biomedclip_ft_lr1e5_ep5.yaml',
     '--indiana-dir', str(INDIANA_DIR)],
    cwd='medical-clip', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
train_log = []
for line in proc.stdout:
    print(line, end='', flush=True)
    train_log.append(line)
proc.wait()
print(f'--- finished in {(time.time()-t0)/60:.1f} min, rc={proc.returncode}')
assert proc.returncode == 0
CKPT = 'checkpoints_biomedclip_lr1e5_ep5/best.pt'

## 2. Retrieval on the OpenI test split

In [ ]:
!cd medical-clip && python3 scripts/stage3_medclip_diagnostic.py \
    --config configs/biomedclip_ft_lr1e5_ep5.yaml \
    --checkpoint {CKPT} --indiana-dir {INDIANA_DIR} \
    --out /kaggle/working/stage5_retrieval.md

## 3. NIH zero-shot, all 7 prompt templates

`--sample 2000` reproduces the exact seed-42 subset used for the published 0.589, so the numbers are
directly comparable. Images are encoded once and reused across templates.

In [ ]:
!cd medical-clip && python3 src/zeroshot.py \
    --config configs/biomedclip_ft_lr1e5_ep5.yaml \
    --checkpoint {CKPT} --nih-dir {NIH_DIR} \
    --prompt all --sample 2000 \
    --out /kaggle/working/stage5_nih_zeroshot.md

## 4. Compare against the published CLIP+ClinicalBERT numbers

In [ ]:
import re

published = {'simple': 0.4487, 'findings': 0.4762, 'clinical': 0.5273, 'patient': 0.5889,
             'radiologist': 0.4854, 'ensemble': 0.4882, 'pos_neg': 0.4496}

rows = Path('/kaggle/working/stage5_nih_zeroshot.md').read_text().splitlines()
new = {}
for line in rows:
    cells = [c.strip() for c in line.strip('|').split('|')]
    if cells and cells[0] in published:
        new[cells[0]] = float(cells[-1])

print(f"{'prompt':<12} {'CLIP+ClinBERT':>14} {'BioMedCLIP':>12} {'delta':>8}")
for k in published:
    d = new[k] - published[k]
    print(f'{k:<12} {published[k]:>14.4f} {new[k]:>12.4f} {d:>+8.4f}')
print()
SINGLE = ['simple', 'findings', 'clinical', 'patient', 'radiologist']

def verdicts(scores, label):
    best_single = max(SINGLE, key=lambda k: scores[k])
    print(f'{label}: best overall={max(scores, key=scores.get)}, best single={best_single}')
    print(f'  ensemble - best single = {scores["ensemble"] - scores[best_single]:+.4f} '
          f'(rank {sorted(scores, key=scores.get, reverse=True).index("ensemble") + 1} of {len(scores)})')
    print(f'  pos_neg is last: {min(scores, key=scores.get) == "pos_neg"}')

verdicts(published, 'CLIP+ClinicalBERT')
verdicts(new, 'BioMedCLIP FT')
print()
print('Compare the two blocks: a claim only generalizes if it holds in both.')
print('Note the ensemble line is a margin, not a boolean -- the sign can hold while')
print('the magnitude collapses into noise.')

## 5. Save outputs

Then **Save Version** so `/kaggle/working` survives the session.

In [ ]:
import shutil
shutil.copy(f'medical-clip/{CKPT}', '/kaggle/working/biomedclip_ft_lr1e5_ep5_best.pt')
with open('/kaggle/working/stage5_train_log.txt', 'w') as f:
    f.write(''.join(train_log))
!rm -rf /kaggle/working/medical-clip/checkpoints_biomedclip_*
!ls -lh /kaggle/working